# Ephemeral RayJob — Ray Data

Topic 2: [ephemeral cluster workflow](https://developers.redhat.com/articles/2025/12/03/tame-ray-workloads-openshift-ai-kuberay-and-kueue#the_ephemeral_cluster__self_service__automated_jobs) — `RayJob` + `ManagedClusterConfig`.

Runs `scale_data.py` on a KubeRay cluster; cluster is deleted when the job finishes.

Official docs: [Running Ray workloads from Jupyter](https://docs.redhat.com/en/documentation/red_hat_openshift_ai_self-managed/3.4/html/working_with_distributed_workloads/running-ray-based-distributed-workloads_distributed-workloads)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from workshop_bootstrap import setup_workshop_path

REPO_ROOT = setup_workshop_path()
from workshop_common import (
    LOCAL_QUEUE,
    NAMESPACE,
    RayJob,
    cpu_cluster_config,
    login,
    print_job_logs,
    repo_root,
    runtime_env_for_script,
    show_local_queues,
    submit_rayjob,
    wait_for_rayjob,
)

print(f"Repo root: {repo_root()}")

In [ ]:
# Token + server: https://docs.redhat.com/en/documentation/red_hat_openshift_ai_self-managed/3.4/html/working_with_distributed_workloads/preparing-the-distributed-training-environment_distributed-workloads#using-the-cluster-server-and-token-to-authenticate_preparing-the-distributed-training-environment
OPENSHIFT_SERVER = "https://api.YOUR_CLUSTER:6443"
OPENSHIFT_TOKEN = "REPLACE_WITH_YOUR_TOKEN".strip()

auth = login(OPENSHIFT_TOKEN, OPENSHIFT_SERVER)
print(f"Namespace: {NAMESPACE}")
print(f"Local queue (default): {LOCAL_QUEUE}")
print(f"Repo root: {repo_root()}")
show_local_queues(NAMESPACE)

In [ ]:
cluster_config = cpu_cluster_config(num_workers=2)

job = RayJob(
    job_name="ray-workshop-scale-data",
    entrypoint="python extras/scripts/scale_data.py",
    cluster_config=cluster_config,
    namespace=NAMESPACE,
    local_queue=LOCAL_QUEUE,
    runtime_env=runtime_env_for_script(
        pip=["pyarrow>=14", "pandas>=2"],
        env_vars={
            "INPUT_PATH": "extras/data/iris.csv",
            "OUTPUT_DIR": "/tmp/ray-workshop-output/iris",
        },
    ),
    ttl_seconds_after_finished=600,
)

submit_rayjob(job)

In [ ]:
wait_for_rayjob(job)
print_job_logs(job)

## Verify in OpenShift AI

1. OpenShift AI → Distributed workloads (or search RayJobs in the console).
2. Confirm `ray-workshop-scale-data` succeeded.
3. Optional CLI from Web Terminal:

```sh
oc get rayjob -n ray-workshop
oc logs -n ray-workshop -l ray.io/node-type=head -c ray-head --tail=80
```

Look for `Done. Wrote N parquet file(s)` in the logs.